# MinbarAI — TranslateGemma QLoRA fine-tune (Colab or Kaggle)

**Colab (recommended for 12B):** Runtime > Change runtime type > **A100 GPU** (or L4). Left sidebar key icon > add secret `HF_TOKEN` (HuggingFace WRITE token) > enable notebook access. ~2-3 h on A100 with bf16.

**Kaggle:** Settings > Accelerator > GPU T4 x2, Internet On, Add-ons > Secrets > `HF_TOKEN`. Runs in fp32 (T4 fp16 overflows on the 12B) — ~18-22 h, needs a resume across two sessions.

Accept the `google/translategemma-12b-it` license on HF once. Set `DRY_RUN = True` for the 4B validation pass (already done), `False` for the 12B mission run. Precision/batch are auto-detected from the GPU.

In [ ]:
DRY_RUN = True   # <-- flip to False for the 12B mission run
HF_USER = "CHANGE_ME"  # your HF username

MODEL = "google/translategemma-4b-it" if DRY_RUN else "google/translategemma-12b-it"
REPO = f"{HF_USER}/translategemma-{'4b' if DRY_RUN else '12b'}-khutbah-lora"

In [ ]:
# 1. Code + deps + HF auth (works on Kaggle AND Colab)
!git clone -b cloud-pipeline https://github.com/Yacine-DH/MinbarAI.git
%cd MinbarAI
!pip install -q -U transformers peft bitsandbytes datasets accelerate rapidfuzz sentencepiece

import os
try:                     # Kaggle: Add-ons > Secrets > HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    ENV = "kaggle"
except ImportError:      # Colab: key icon (left sidebar) > add HF_TOKEN, enable notebook access
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    ENV = "colab"

from huggingface_hub import login
login(os.environ["HF_TOKEN"])

import torch
BF16 = torch.cuda.is_bf16_supported()
PRECISION = "bf16" if BF16 else "fp32"
GPU = torch.cuda.get_device_name(0)
BATCH, ACCUM = (4, 4) if BF16 else (1, 16)
print(f"env={ENV} gpu={GPU} precision={PRECISION} batch={BATCH}x{ACCUM}")

In [ ]:
# 2. Build the dataset (~24k pairs, eval cases excluded)
!python training/build_dataset.py

In [ ]:
# 3. Train (resumable — checkpoints push to the Hub every save)
!python training/finetune_qlora.py --model $MODEL --hub-repo $REPO --precision $PRECISION --batch $BATCH --grad-accum $ACCUM

In [ ]:
# 4. Merge adapter into the base model and push the merged model
!pip install -q -U torchao   # Kaggle ships 0.10, peft needs >=0.16
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

merged_repo = REPO.replace("-lora", "")
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cpu")
model = PeftModel.from_pretrained(base, REPO)
model = model.merge_and_unload()
tok = AutoTokenizer.from_pretrained(MODEL)
model.push_to_hub(merged_repo)
tok.push_to_hub(merged_repo)
print("merged ->", merged_repo)

In [ ]:
# 5. GGUF Q4_K_M + push (for Ollama serving on Modal)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!hf download {merged_repo} --local-dir merged_hf
!python llama.cpp/convert_hf_to_gguf.py merged_hf --outfile model-f16.gguf --outtype f16
!cmake -B llama.cpp/build llama.cpp -DGGML_CUDA=OFF && cmake --build llama.cpp/build --target llama-quantize -j4
!llama.cpp/build/bin/llama-quantize model-f16.gguf model-Q4_K_M.gguf Q4_K_M
!hf upload {merged_repo} model-Q4_K_M.gguf model-Q4_K_M.gguf
print("GGUF pushed. Serve on Modal by setting MODEL_ID to an Ollama model created FROM this GGUF (see KAGGLE_MISSION.md step 5).")